<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt



In [2]:
df = pd.read_csv("ccolli78_project_summary.csv", sep=";", header = None)
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

,project,commit,author,time,message,Month
0,caseyyoungflesh_post,0043f9cf35898788d2fe031c895320f5a95c4b5d,Casey Youngflesh <caseyyoungflesh@gmail.com>,1557198520,Merge branch 'master' of github.com:caseyyoung...,2019-05
1,caseyyoungflesh_post,01539dd71fa1e93fa064cf775bcb4e2d02f54f55,Casey Youngflesh <caseyyoungflesh@gmail.com>,1512592109,Fix doc notes\n,2017-12
2,caseyyoungflesh_post,015d652a117bbfe353630d7b4abbd823bc2dc0c2,Casey Youngflesh <caseyyoungflesh@gmail.com>,1553985422,MCMCtrace - plot = FALSE\n,2019-03
3,caseyyoungflesh_post,0277d11bcb3f5d4df3ee6d0ce2ede145d54a332c,Casey Youngflesh <casey.youngflesh@stonybrook....,1480452033,Clean up code\n,2016-11
4,caseyyoungflesh_post,0303714ef930f30fe442dd2ede35663cafd0fb89,Casey Youngflesh <caseyyoungflesh@gmail.com>,1503709866,More rearrangement of code\n,2017-08


In [3]:
import matplotlib.pyplot as plt

def make_timeseries(prj):
    df1 = df[df['project'] == prj]
    monthly_counts = df1.groupby('Month').size().reset_index(name='Commits')
    monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'])
    full_date_range = pd.DataFrame({'time': pd.date_range(start=monthly_counts['time'].min(),
                                                          end=monthly_counts['time'].max(), freq='MS')})
    monthly_counts = pd.merge(full_date_range, monthly_counts, on='time', how='left').fillna(0)
    monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')
    monthly_counts['Commits'] = monthly_counts['Commits'].astype(int)

    # CSV: Month;#Commits
    monthly_counts[['Month', 'Commits']].rename(columns={'Commits': '#Commits'}) \
        .to_csv(f"ccolli78_commits_timeseries_{prj}.csv", sep=';', index=False)

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(monthly_counts['Month'], monthly_counts['Commits'], marker='o', linestyle='-')
    plt.xlabel('Time (YYYY-MM)')
    plt.ylabel('Number of Commits')
    plt.title(prj)
    plt.grid(True)
    ax = plt.gca()
    for label in ax.xaxis.get_ticklabels()[::2]:
        label.set_visible(False)
    plt.xticks(rotation=45, fontsize=7)
    plt.tight_layout()
    plt.savefig(f"ccolli78_timeseries_{prj}.png", dpi=300)
    plt.close()
    return monthly_counts

results = {}
for prj in df['project'].unique():
    results[prj] = make_timeseries(prj)
    print(prj, len(results[prj]), 'months')


caseyyoungflesh_post 64 months


jackwasey_icd 127 months


oxfordcontrol_cosmo.jl 93 months


jbei_edd 121 months


davidcarslaw_openair 131 months


qutech-delft_quantuminspire 89 months


tryolabs_norfair 63 months


soft-matter_trackpy 162 months


cosmoscout_cosmoscout-vr 86 months


scify_jedaitoolkit 113 months


In [4]:
def gap_stats(mc, min_gap=3):
    counts = mc['Commits'].tolist()
    months = mc['Month'].tolist()
    runs = []                      # (start_index, end_index) of each zero run
    start = None
    for i, c in enumerate(counts + [1]):   # sentinel so a trailing run closes
        if c == 0 and start is None:
            start = i
        elif c != 0 and start is not None:
            runs.append((start, i - 1))
            start = None
    if not runs:
        return {'LongestGapStart': None, 'LongestGapEnd': None, 'LongestGapLength': 0,
                'NumCommitsAfterLastGap': sum(counts), 'NumberOfGapsInTimeline': 0}
    s, e = max(runs, key=lambda r: r[1] - r[0])
    return {'LongestGapStart': months[s],
            'LongestGapEnd': months[e],
            'LongestGapLength': e - s + 1,
            'NumCommitsAfterLastGap': sum(counts[e + 1:]),
            'NumberOfGapsInTimeline': sum(1 for a, b in runs if b - a + 1 >= min_gap)}

gaps = pd.DataFrame({prj: gap_stats(mc) for prj, mc in results.items()}).T
gaps


,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
caseyyoungflesh_post,2016-05,2016-08,4,667,4
jackwasey_icd,2021-12,2024-11,36,1,3
oxfordcontrol_cosmo.jl,2023-11,2024-03,5,52,2
jbei_edd,2017-12,2017-12,1,1318,0
davidcarslaw_openair,2021-04,2021-08,5,1093,1
qutech-delft_quantuminspire,2019-12,2020-03,4,1050,1
tryolabs_norfair,2021-08,2021-09,2,893,0
soft-matter_trackpy,2020-09,2021-01,5,212,5
cosmoscout_cosmoscout-vr,2019-04,2019-06,3,5431,1
scify_jedaitoolkit,2024-05,2025-06,14,2,7


In [5]:
gh = {  # stars, forks, last GH commit (from Part 1, read 2026-09-30)
    'caseyyoungflesh_post': (39, 4, '2025-11-25'),
    'cosmoscout_cosmoscout-vr': (428, 45, '2026-09-07'),
    'davidcarslaw_openair': (365, 123, '2026-09-28'),
    'jackwasey_icd': (259, 63, '2026-09-14'),
    'jbei_edd': (16, 12, '2025-01-09'),
    'oxfordcontrol_cosmo.jl': (319, 46, '2026-07-07'),
    'qutech-delft_quantuminspire': (75, 30, '2026-09-25'),
    'scify_jedaitoolkit': (226, 43, '2025-07-12'),
    'soft-matter_trackpy': (510, 149, '2026-06-29'),
    'tryolabs_norfair': (2688, 277, '2025-04-30'),
}
pattern = {  # from the plots: steady, rising, declining, U-shaped, cyclical, irregular
    'caseyyoungflesh_post': 'irregular',
    'cosmoscout_cosmoscout-vr': 'declining',
    'davidcarslaw_openair': 'U-shaped',
    'jackwasey_icd': 'declining',
    'jbei_edd': 'declining',
    'oxfordcontrol_cosmo.jl': 'declining',
    'qutech-delft_quantuminspire': 'U-shaped',
    'scify_jedaitoolkit': 'declining',
    'soft-matter_trackpy': 'declining',
    'tryolabs_norfair': 'declining',
}

stats = df.groupby('project').agg(ncommits=('commit', 'count'), nauthors=('author', 'nunique'),
                                  **{'from': ('time', 'min'), 'to': ('time', 'max')})
stats['nstars'] = [gh[p][0] for p in stats.index]
stats['nforks'] = [gh[p][1] for p in stats.index]
stats['lastGHCommitDate'] = [gh[p][2] for p in stats.index]
stats = stats.join(gaps)
stats['ActivityPattern'] = [pattern[p] for p in stats.index]
stats = stats[['ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate',
               'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap',
               'ActivityPattern', 'NumberOfGapsInTimeline']]
stats.index.name = 'Project'
stats.to_csv('ccolli78_project_stats.csv', sep=';')
stats

,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline
Project,,,,,,,,,,,,,
caseyyoungflesh_post,724,8,1458715949,1624558029,39,4,2025-11-25,2016-05,2016-08,4,667,irregular,4
cosmoscout_cosmoscout-vr,5442,67,1551714942,1775135410,428,45,2026-09-07,2019-04,2019-06,3,5431,declining,1
davidcarslaw_openair,2014,35,1421337704,1762281919,365,123,2026-09-28,2021-04,2021-08,5,1093,U-shaped,1
jackwasey_icd,3138,36,1401671991,1734714091,259,63,2026-09-14,2021-12,2024-11,36,1,declining,3
jbei_edd,4855,28,1418339301,1733890409,16,12,2025-01-09,2017-12,2017-12,1,1318,declining,0
oxfordcontrol_cosmo.jl,1296,31,1514998395,1758823682,319,46,2026-07-07,2023-11,2024-03,5,52,declining,2
qutech-delft_quantuminspire,1377,50,1532959943,1762546596,75,30,2026-09-25,2019-12,2020-03,4,1050,U-shaped,1
scify_jedaitoolkit,658,29,1458130700,1752295364,226,43,2025-07-12,2024-05,2025-06,14,2,declining,7
soft-matter_trackpy,2421,68,1340299131,1762541155,510,149,2026-06-29,2020-09,2021-01,5,212,declining,5


# Interpretation

Gaps below are runs of at least three consecutive zero-commit months.

- **caseyyoungflesh_post** (irregular): short bursts of activity (a 119-commit spike in 2016-09) separated by quiet stretches; WoC data stops in 2021-06 although GitHub shows commits to 2025. 4 gaps; longest 2016-05 to 2016-08 (4 months).
- **cosmoscout_cosmoscout-vr** (declining): very heavy development in 2019-2020 (200+ commits/month), then a steady lower level of 10-100/month through 2026. 1 gap; longest 2019-04 to 2019-06 (3 months).
- **davidcarslaw_openair** (U-shaped): active 2015-2016, low single-digit months 2017-2021, then a clear revival from 2022 onward with the second-highest peak in 2025-02. 1 gap; longest 2021-04 to 2021-08 (5 months).
- **jackwasey_icd** (declining): cycles of intense work (2014-2016, 2018-2020) followed by 36 months of silence from 2021-12 to 2024-11 and a single commit after. 3 gaps.
- **jbei_edd** (declining): peak of ~200 commits/month in 2016-2017, then a long slow decline to a trickle by 2024. 0 gaps; longest zero stretch is 1 month (2017-12).
- **oxfordcontrol_cosmo.jl** (declining): strong 2018-2020, tapering to occasional small commits since 2021. 2 gaps; longest 2023-11 to 2024-03 (5 months).
- **qutech-delft_quantuminspire** (U-shaped): 2019 peak, quiet 2020-2022, then rising steadily to a new high in 2025. 1 gap; longest 2019-12 to 2020-03 (4 months).
- **scify_jedaitoolkit** (declining): bursts in 2017, 2018 and 2020, then almost nothing after 2022; the repo is now archived on GitHub. 7 gaps; longest 2024-05 to 2025-06 (14 months).
- **soft-matter_trackpy** (declining): most activity 2012-2016, low but continuous maintenance since, with several short gaps. 5 gaps; longest 2020-09 to 2021-01 (5 months).
- **tryolabs_norfair** (declining): two large release bursts (2020-08, 2022-07), then small maintenance commits. 0 gaps; longest zero stretch is 2 months (2021-08 to 2021-09).

# Part 3

In [6]:
# Part 3.1: last 10 commits before and first 10 after each project's longest gap
dfs = df.sort_values('time')
rows = []
for prj, s in stats.iterrows():
    d = dfs[dfs['project'] == prj]
    rows.append(d[d['Month'] < s['LongestGapStart']].tail(10).assign(prepost='pre'))
    rows.append(d[d['Month'] > s['LongestGapEnd']].head(10).assign(prepost='post'))
gap_commits = pd.concat(rows)[['prepost', 'project', 'commit', 'author', 'time', 'message']]
gap_commits.to_csv('ccolli78_project_gap_commits.csv', sep=';', index=False)
gap_commits.groupby(['project', 'prepost']).size().unstack()

prepost,post,pre
project,,
caseyyoungflesh_post,10,10
cosmoscout_cosmoscout-vr,10,10
davidcarslaw_openair,10,10
jackwasey_icd,1,10
jbei_edd,10,10
oxfordcontrol_cosmo.jl,10,10
qutech-delft_quantuminspire,10,10
scify_jedaitoolkit,2,10
soft-matter_trackpy,10,10


In [7]:
# Part 3: themes, gap interpretation, current status (evidence in the reflection .md files)
part3 = {
    "caseyyoungflesh_post": {
        "BeforeThemes": "Bug fixes:Documentation updates",
        "AfterThemes": "Feature development:Bug fixes",
        "HypothesizedGapReason": "Single-author project (nearly all commits by Casey Youngflesh). The 10 commits before the 2016-05..2016-08 gap are small fixes and doc notes with nothing announcing a pause, and no GitHub issues or PRs exist from 2016. Most likely a short pause by the sole developer, not abandonment.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed 2016-09-27 with a 119-commit month adding a second posterior plot style (poplot2) and plot options; a feature push by the author.",
        "WhoRecovered": "Same person (Casey Youngflesh)",
        "Currentstatus": "Active",
        "RecentThemes": "Release:Feature development",
        "Notes": "Short bursty single-author project; the longest gap is only 4 months in summer 2016 and was easy to read as a pause rather than abandonment. Activity came back with a large feature burst by the same author. WoC data stops in 2021-06 but GitHub (repo renamed to MCMCvis) shows CRAN-prep commits as late as 2025-11-25."
    },
    "cosmoscout_cosmoscout-vr": {
        "BeforeThemes": "Other:Documentation updates",
        "AfterThemes": "Other:Documentation updates",
        "HypothesizedGapReason": "The commits before the 2019-04..2019-06 gap (March 2019) are all logo and icon artwork by Simon Schneegans; the commits after it (2019-07-15) are the initial repository scaffolding (CLA, contribution guide, .gitignore, clang-format, build scripts, README). The gap looks like the quiet period between creating the project identity and preparing the public open-source repository, not a stall in development.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Public repository set up in July 2019, followed by sustained development of 100-280 commits/month through 2020.",
        "WhoRecovered": "Same person at first (Simon Schneegans); the project later grew to 67 authors, and recent commits are by Jonas Gilg.",
        "Currentstatus": "Active",
        "RecentThemes": "Feature development:Other",
        "Notes": "The only 3-month gap sits right at the start of the repository, between logo work and the July 2019 public scaffolding commits, so it is a setup artifact rather than inactivity. Since then the project has never had a zero-commit month, though volume has declined from the 2019-2020 peak. Still active in 2026 with feature and build-system work."
    },
    "davidcarslaw_openair": {
        "BeforeThemes": "Bug fixes:Documentation updates",
        "AfterThemes": "Feature development:Release",
        "HypothesizedGapReason": "From 2017 to 2021 the project was effectively maintained by one person (davidcarslaw). Commit messages give no reason for the 2021-04..2021-08 pause. About 12 GitHub issues were opened by users during the gap (questions and bug reports, nearly all closed later), so users were active but nobody was committing. Likely a maintainer-availability pause.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Commits resumed 2021-09-14 with a new plot option and a commit 'for CRAN release' the next day, so a CRAN release drove the restart. From 2022 activity rose sharply, which coincides with Jack Davison becoming the main contributor (most 2025-2026 commits are his).",
        "WhoRecovered": "Same person restarted it (davidcarslaw); the later growth is driven by a new contributor (Jack Davison). Repo has since moved to the openair-project organisation.",
        "Currentstatus": "Active",
        "RecentThemes": "Bug fixes:Feature development",
        "Notes": "U-shaped: busy 2015-2016, a long low period 2017-2021 including the 5-month gap, then a strong revival from 2022. The gap itself was hard to explain from commits (no message mentions it), but the user issues kept coming, pointing to maintainer time rather than loss of interest. Recovery was started by the original author and sustained by a new maintainer."
    },
    "jackwasey_icd": {
        "BeforeThemes": "Other:Dependency updates",
        "AfterThemes": "Documentation updates",
        "HypothesizedGapReason": "The last commits before the 36-month gap (March 2021) are all about getting the package through R-hub/CRAN checks ('rhub wip', dropping xml2/RODBC dependencies), followed by a lone CodeQL workflow in 2021-11. The GitHub description now reads '*New maintainer/owner needed*', and the 4 issues opened during the gap (bug reports and install failures, 2021-12 to 2023-03) are still open with no maintainer reply. Evidence points to the sole maintainer stepping away after a difficult CRAN-check cycle.",
        "HasRecovered": "Partially",
        "WhyRecovered": "A single README commit on 2024-12-20 ended the gap. In 2026-09 the same author made a burst of commits (PR #219 'fix-modern-r-check': GitHub Actions instead of Travis/AppVeyor, R-hub v2, snprintf, S3 method warnings), i.e. a push to make the package pass checks on modern R.",
        "WhoRecovered": "Same person (Jack Wasey); no new maintainer has appeared despite the request in the repo description.",
        "Currentstatus": "Active",
        "RecentThemes": "Other:Bug fixes",
        "Notes": "Clear case of a single-maintainer R package going dormant: 36 zero-commit months from 2021-12 to 2024-11 after a run of CRAN-check commits, unanswered issues, and a 'new maintainer needed' notice. The gap was easy to interpret from this evidence. Only one commit followed the gap in WoC data, but GitHub shows the original author returned in 2026-09 to modernise the build and checks."
    },
    "jbei_edd": {
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Bug fixes:Other",
        "HypothesizedGapReason": "N/A active Project (longest zero-commit stretch is a single month, 2017-12; there is no gap of 3+ months).",
        "HasRecovered": "N/A (no gap)",
        "WhyRecovered": "N/A",
        "WhoRecovered": "N/A",
        "Currentstatus": "Inactive",
        "RecentThemes": "Bug fixes:Other",
        "Notes": "No real gap, but a steady decline from about 200 commits/month in 2016-2017 to a trickle by 2024, with the two main authors (William Morrell, Mark Forrer) narrowing to just Morrell. The last GitHub commit is 2024-12-11, more than six months before 2025-09-30, so the project counts as inactive now. Recent commits are maintenance: migration and test fixes, removing a dependency."
    },
    "oxfordcontrol_cosmo.jl": {
        "BeforeThemes": "Feature development:Automated bot contributions",
        "AfterThemes": "Documentation updates:Automated bot contributions",
        "HypothesizedGapReason": "Before the 2023-11..2024-03 gap the only human commits (2023-07 to 2023-09) were by outside contributors (Benoit Legat, Oscar Dowson) adapting the MathOptInterface wrapper; the core maintainers made no commits, and the last pre-gap activity was a CompatHelper bot PR. Activity had been low since 2021. Looks like maintainer inattention on a mature solver rather than a specific event.",
        "HasRecovered": "Yes (modestly, 52 commits after)",
        "WhyRecovered": "A crash fix by Paul Goulart (PR #187, 2024-04) was followed by Michael Garstka adding documentation sections, a how-to-develop guide and release 0.8.9 (2024-05). Releases have continued (0.8.10 to 0.8.12 in 2026).",
        "WhoRecovered": "Same core maintainers (Michael Garstka, Paul Goulart) plus occasional external PRs.",
        "Currentstatus": "Active",
        "RecentThemes": "Release:Bug fixes",
        "Notes": "Declining from a strong 2018-2020 period into low maintenance, with the longest gap (5 months) in late 2023 when only outside contributors and bots were touching the repo. The gap was moderately easy to read: no stated reason, but the pattern of external PRs sitting without maintainer commits is clear. Maintainers returned in 2024 with a fix, docs and a release, and keep shipping small version bumps."
    },
    "qutech-delft_quantuminspire": {
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Bug fixes:Documentation updates",
        "HypothesizedGapReason": "Commits carry Jira-style ticket ids ([DEM-xxxx]) from a small QuTech team; one pre-gap PR is '[DEM-1183] Offboard team member' (2019-10). No PRs or issues at all were created between 2019-11-19 and 2020-04-03. No reason is stated in messages; most likely a team-capacity or prioritisation pause over the 2019-2020 winter.",
        "HasRecovered": "Yes",
        "WhyRecovered": "April 2020 brought a burst of ticketed fixes, hardware-backend support and the 1.2.0 release (PR 2020-04-17). From 2023 activity rose steadily to a new high in 2025; recent commits use a [QI2-xxxx] prefix, suggesting work for a new Quantum Inspire 2 platform.",
        "WhoRecovered": "Same developer (QFer) restarted it in 2020; the 2023-2025 rise comes from new people (e.g. Nischal Sehrawat) plus Dependabot.",
        "Currentstatus": "Active",
        "RecentThemes": "Automated bot contributions:Bug fixes",
        "Notes": "U-shaped: 2019 peak, quiet 2020-2022 (including the 4-month longest gap), then rising to a new high in 2025. The gap was hard to interpret because the messages are ticket numbers with no narrative; the only hint is a team-member offboarding just before it. Recovery was first the same team member, later new contributors; today most commits are Dependabot bumps."
    },
    "scify_jedaitoolkit": {
        "BeforeThemes": "Documentation updates:Feature development",
        "AfterThemes": "Automated bot contributions:Dependency updates",
        "HypothesizedGapReason": "Activity had been sparse since 2022 (7 gaps of 3+ months). The last human commit before the 14-month gap is 'Add files via upload' (gpapadis, 2024-04-12). Only one issue was opened during the gap ('Automation analysis', 2025-04, still open with no reply). The repository is now archived on GitHub. The maintainers appear to have wound the project down.",
        "HasRecovered": "No",
        "WhyRecovered": "N/A. The only post-gap activity is a Dependabot security bump of commons-lang3 merged on 2025-07-12; nothing since, and the repo is archived.",
        "WhoRecovered": "N/A (the bot PR was merged by Paul Isaris, not the earlier maintainer gpapadis, but there is no ongoing contributor).",
        "Currentstatus": "Active",
        "RecentThemes": "Other:Dependency updates",
        "Notes": "Declining to zero: bursts in 2017, 2018 and 2020, then almost nothing, with the longest gap (14 months, 2024-05 to 2025-06) ended only by a Dependabot merge. Easy to interpret as abandonment given the archived status and the unanswered 2025 issue. By the six-month rule the last commit (2025-07-12) still counts as Active, but the repo is read-only and should be considered dead."
    },
    "soft-matter_trackpy": {
        "BeforeThemes": "Bug fixes:Other",
        "AfterThemes": "Other:Dependency updates",
        "HypothesizedGapReason": "Before the 2020-09..2021-01 gap, a contributor's PRs (Maxim Ivanov, 2020-08) were left in CI-merge state and were only merged in 2021-02 (#632). About 8 user issues were opened during the gap, several still open. The maintainers (Nathan Keim, Casper van der Wel) simply did not commit for five months on a mature project; no reason is stated.",
        "HasRecovered": "Yes (212 commits after, as low-level maintenance)",
        "WhyRecovered": "In 2021-02 Casper van der Wel moved CI to GitHub Actions, fixed NumPy 1.20 / SciPy 1.5 deprecations and version constraints, dropped Python 2 and merged the pending PRs; a user then asked for a release (issue 2021-03-17). Recovery was driven by ecosystem breakage, not new features.",
        "WhoRecovered": "Same maintainers (van der Wel, Keim) plus merged external PRs (Ivanov, hugovk).",
        "Currentstatus": "Active",
        "RecentThemes": "Documentation updates:Feature development",
        "Notes": "Long-lived library (since 2012) in maintenance mode: most work was 2012-2016, with five 3+ month gaps since 2020. The longest gap was moderately easy to read: pending contributor PRs and user issues piled up while the maintainers were away, and the restart was all CI and dependency repair. Still active in 2026 with Nathan Keim merging small PRs."
    },
    "tryolabs_norfair": {
        "BeforeThemes": "Release:Documentation updates",
        "AfterThemes": "Feature development:Bug fixes",
        "HypothesizedGapReason": "N/A active Project (longest zero-commit stretch is 2 months, 2021-08 to 2021-09; no gap of 3+ months).",
        "HasRecovered": "N/A (no gap)",
        "WhyRecovered": "N/A",
        "WhoRecovered": "N/A",
        "Currentstatus": "Active",
        "RecentThemes": "Release:Dependency updates",
        "Notes": "No real gap, but a declining pattern: two large release bursts (243 commits in 2020-08, 205 in 2022-07) and small maintenance since 2023. The last GitHub commits (2025-04-30) are a v2.3.0 maintenance release by javiber relaxing dependency pins and updating CI, which just clears the six-month Active cutoff. Note that WoC holds many commits twice with different SHAs (rebased copies), which inflates its count versus GitHub."
    }
}

for col in ['BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered',
            'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']:
    stats[col] = [part3[p][col] for p in stats.index]
stats.to_csv('ccolli78_project_stats.csv', sep=';')
stats[['LongestGapLength', 'BeforeThemes', 'AfterThemes', 'HasRecovered', 'Currentstatus', 'RecentThemes']]

,LongestGapLength,BeforeThemes,AfterThemes,HasRecovered,Currentstatus,RecentThemes
Project,,,,,,
caseyyoungflesh_post,4,Bug fixes:Documentation updates,Feature development:Bug fixes,Yes,Active,Release:Feature development
cosmoscout_cosmoscout-vr,3,Other:Documentation updates,Other:Documentation updates,Yes,Active,Feature development:Other
davidcarslaw_openair,5,Bug fixes:Documentation updates,Feature development:Release,Yes,Active,Bug fixes:Feature development
jackwasey_icd,36,Other:Dependency updates,Documentation updates,Partially,Active,Other:Bug fixes
jbei_edd,1,Bug fixes:Feature development,Bug fixes:Other,N/A (no gap),Inactive,Bug fixes:Other
oxfordcontrol_cosmo.jl,5,Feature development:Automated bot contributions,Documentation updates:Automated bot contributions,"Yes (modestly, 52 commits after)",Active,Release:Bug fixes
qutech-delft_quantuminspire,4,Bug fixes:Feature development,Bug fixes:Documentation updates,Yes,Active,Automated bot contributions:Bug fixes
scify_jedaitoolkit,14,Documentation updates:Feature development,Automated bot contributions:Dependency updates,No,Active,Other:Dependency updates
soft-matter_trackpy,5,Bug fixes:Other,Other:Dependency updates,"Yes (212 commits after, as low-level maintenance)",Active,Documentation updates:Feature development
